In [2]:
import torch
import torchvision.transforms as tvt
from torch.utils.data import DataLoader
from tqdm import tqdm
import shutil
import pandas as pd
import numpy as np
import os
torch.manual_seed(1234)
from evaluate_metrics import compute_eer
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
font = {'family' : 'normal',
        'weight' : 'normal',
        'size'   : 14}

def recursive_file_list(root_dir, ext):
    return [os.path.join(dp, f) for dp, dn, filenames in os.walk(root_dir) for f in filenames if f.endswith(ext)]



/home/phucdt/anaconda3/envs/bio/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# LA19

In [3]:
# load eval set
eval_df = pd.read_csv("/datab/Dataset/ASVspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt", sep=" ", header=None)
eval_df.columns = ["sid", "utt","phy", "attack", "label"]
eval_df.head


<bound method NDFrame.head of            sid           utt phy attack     label
0      LA_0039  LA_E_2834763   -    A11     spoof
1      LA_0014  LA_E_8877452   -    A14     spoof
2      LA_0040  LA_E_6828287   -    A16     spoof
3      LA_0022  LA_E_6977360   -    A09     spoof
4      LA_0031  LA_E_5932896   -    A13     spoof
...        ...           ...  ..    ...       ...
71232  LA_0004  LA_E_1665632   -      -  bonafide
71233  LA_0038  LA_E_5085671   -    A09     spoof
71234  LA_0012  LA_E_4926022   -    A16     spoof
71235  LA_0052  LA_E_2894498   -      -  bonafide
71236  LA_0009  LA_E_4689563   -    A19     spoof

[71237 rows x 5 columns]>

## CNNbio RawNet2 TTS only

In [6]:
# load score files
score_files = recursive_file_list("./docs/cnnbio_tts_only_train_res", ".txt")
# filter out the la19 files
score_files = [f for f in score_files if "la19" in f]
score_files = sorted(score_files)

score_df = pd.DataFrame()
score_df["system"] = ["_".join(f.split("/")[-1].split("_")[1:4]) for f in score_files]
score_df["eer"] = np.nan
score_df[[f'A{_id:02d}' for _id in range(7, 20)]] = np.nan
# print(score_df)
for score_file in score_files:
    system = "_".join(score_file.split("/")[-1].split("_")[1:4])
    print(f"Processing {system}")
    scores = pd.read_csv(score_file, sep=' ', header=None)
    scores.columns = ['utt', 'www', 'aaa', 'score']
    # print(scores.head())
    scores = scores.merge(eval_df, on=["utt"])
    bona_scores = scores[scores["label"] == "bonafide"]['score']
    spoof_scores = scores[scores["label"] == "spoof"]['score']
    pool_eer, _ = compute_eer(bona_scores, spoof_scores)
    # print(pool_eer)
    # update the score_df
    score_df.loc[score_df["system"] == system, "eer"] = pool_eer
    # break down as attack
    for attack in scores["attack"].unique():
        if attack == "-":
            continue
        attack_scores = scores[scores["attack"] == attack]['score']
        attack_eer, _ = compute_eer(bona_scores, attack_scores)
        # update the score_df
        score_df.loc[score_df["system"] == system, f"{attack}"] = attack_eer
        
print(score_df)
score_df.to_csv("docs/cnnbio_tts_only_train_res/la19_eer.csv", index=False)



Empty DataFrame
Columns: [system, eer, A07, A08, A09, A10, A11, A12, A13, A14, A15, A16, A17, A18, A19]
Index: []


### CNNbio W2V gMLP

In [5]:
# load score files
score_files = recursive_file_list("./docs/w2v_cnnbio", ".txt")
# filter out the la19 files
score_files = [f for f in score_files if "la19" in f]
score_files = sorted(score_files)

score_df = pd.DataFrame()
score_df["system"] = ["_".join(f.split("/")[-1].split("_")[1:4]) for f in score_files]
score_df["eer"] = np.nan
score_df[[f'A{_id:02d}' for _id in range(7, 20)]] = np.nan
# print(score_df)
for score_file in score_files:
    system = "_".join(score_file.split("/")[-1].split("_")[1:4])
    print(f"Processing {system}")
    scores = pd.read_csv(score_file, sep=' ', header=None)
    scores.columns = ['utt', 'www', 'aaa', 'score']
    # print(scores.head())
    scores = scores.merge(eval_df, on=["utt"])
    bona_scores = scores[scores["label"] == "bonafide"]['score']
    spoof_scores = scores[scores["label"] == "spoof"]['score']
    pool_eer, _ = compute_eer(bona_scores, spoof_scores)
    # print(pool_eer)
    # update the score_df
    score_df.loc[score_df["system"] == system, "eer"] = pool_eer
    # break down as attack
    for attack in scores["attack"].unique():
        if attack == "-":
            continue
        attack_scores = scores[scores["attack"] == attack]['score']
        attack_eer, _ = compute_eer(bona_scores, attack_scores)
        # update the score_df
        score_df.loc[score_df["system"] == system, f"{attack}"] = attack_eer
        
print(score_df)
score_df.to_csv("./docs/w2v_cnnbio/la19_eer.csv", index=False)



Processing W2V_CNNS2S_32concat
                system       eer       A07  A08  A09       A10       A11  \
0  W2V_CNNS2S_32concat  0.057358  0.000985  0.0  0.0  0.003667  0.002682   

        A12  A13      A14       A15       A16       A17       A18       A19  
0  0.002275  0.0  0.00017  0.002207  0.000577  0.160329  0.148988  0.259032  


## GMMbio RawNet2 TTS only


In [4]:
# load score files
score_files = recursive_file_list("./docs/gmmbio_tts_only_train_res", ".txt")
# filter out the la19 files
score_files = [f for f in score_files if "la19" in f]
score_files = sorted(score_files)

score_df = pd.DataFrame()
score_df["system"] = ["_".join(f.split("/")[-1].split("_")[1:4]) for f in score_files]
score_df["eer"] = np.nan
score_df[[f'A{_id:02d}' for _id in range(7, 20)]] = np.nan
# print(score_df)
for score_file in score_files:
    system = "_".join(score_file.split("/")[-1].split("_")[1:4])
    print(f"Processing {system}")
    scores = pd.read_csv(score_file, sep=' ', header=None)
    scores.columns = ['utt', 'www', 'aaa', 'score']
    # print(scores.head())
    scores = scores.merge(eval_df, on=["utt"])
    bona_scores = scores[scores["label"] == "bonafide"]['score']
    spoof_scores = scores[scores["label"] == "spoof"]['score']
    pool_eer, _ = compute_eer(bona_scores, spoof_scores)
    # print(pool_eer)
    # update the score_df
    score_df.loc[score_df["system"] == system, "eer"] = pool_eer
    # break down as attack
    for attack in scores["attack"].unique():
        if attack == "-":
            continue
        attack_scores = scores[scores["attack"] == attack]['score']
        attack_eer, _ = compute_eer(bona_scores, attack_scores)
        # update the score_df
        score_df.loc[score_df["system"] == system, f"{attack}"] = attack_eer
        
print(score_df)
score_df.to_csv("docs/gmmbio_tts_only_train_res/la19_eer.csv", index=False)


Processing RawNet_CNNS2S_32concat


Processing RawNet_CNNS2S_64concat
Processing RawNet_CNNS2S_add
Processing RawNet_RNN_32concat
Processing RawNet_RNN_64concat
Processing RawNet_RNN_add
Processing RawNet_Trans_32concat
Processing RawNet_Trans_64concat
Processing RawNet_Trans_add
                   system       eer       A07       A08       A09       A10  \
0  RawNet_CNNS2S_32concat  0.154862  0.003090  0.013683  0.002275  0.003837   
1  RawNet_CNNS2S_64concat  0.171753  0.002037  0.009201  0.001800  0.003022   
2       RawNet_CNNS2S_add  0.169961  0.002445  0.004889  0.002445  0.003497   
3     RawNet_RNN_32concat  0.173351  0.002445  0.006519  0.002614  0.003497   
4     RawNet_RNN_64concat  0.151164  0.001800  0.011578  0.001222  0.003837   
5          RawNet_RNN_add  0.161282  0.003667  0.011171  0.001460  0.004889   
6   RawNet_Trans_32concat  0.164617  0.001630  0.008319  0.002037  0.003429   
7   RawNet_Trans_64concat  0.167631  0.002207  0.007911  0.002037  0.003837   
8        RawNet_Trans_add  0.161658  0.00227